# A3: Predicting Car Price

## Setup

In [1]:
import numpy as np
import pandas as pd
import sklearn

# Same seed as A2, so the train and test rows are exactly the same cars.
RANDOM_STATE = 42

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)

numpy: 2.4.1
pandas: 3.0.5
sklearn: 1.8.0


## Load and clean the data

These are the same cleaning steps as A2, in the same order.

In [2]:
df = pd.read_csv('Cars.csv')
print("The dataset has", df.shape[0], "rows and", df.shape[1], "columns")

The dataset has 8128 rows and 13 columns


In [3]:
# Remove exact duplicate rows, so the same car cannot land in both train and test.
df = df.drop_duplicates()

# Turn owner from text into numbers 1 to 5 (the categories have a real order).
owner_map = {
    "First Owner": 1,
    "Second Owner": 2,
    "Third Owner": 3,
    "Fourth & Above Owner": 4,
    "Test Drive Car": 5
}
df['owner'] = df['owner'].map(owner_map)

# CNG and LPG use different mileage units (km/kg), so drop them.
df = df[~df['fuel'].isin(['CNG', 'LPG'])]

def strip_unit_to_float(series):
    """Turn text like "23.4 kmpl" into the number 23.4. Bad values become NaN."""
    first_part = series.str.split().str[0]
    return pd.to_numeric(first_part, errors='coerce')

# Remove the units from mileage, engine and max_power.
for col in ['mileage', 'engine', 'max_power']:
    df[col] = strip_unit_to_float(df[col])
    # A zero here is a recording error, so mark it as missing. The imputer fills it later.
    df[col] = df[col].replace(0, np.nan)

# Keep only the first word of name (the brand), then drop torque.
df = df.rename(columns={'name': 'brand'})
df['brand'] = df['brand'].str.split().str[0]
df = df.drop(columns=['torque'])

# Remove the Test Drive Cars (owner = 5), then renumber the rows.
df = df[df['owner'] != 5]
df = df.reset_index(drop=True)

print("Final shape after cleaning:", df.shape)
df.head()

Final shape after cleaning: (6827, 12)


,brand,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,Maruti,2014,450000,145500,Diesel,Individual,Manual,1,23.40,1248.0,74.00,5.0
1,Skoda,2014,370000,120000,Diesel,Individual,Manual,2,21.14,1498.0,103.52,5.0
2,Honda,2006,158000,140000,Petrol,Individual,Manual,3,17.70,1497.0,78.00,5.0
3,Hyundai,2010,225000,127000,Diesel,Individual,Manual,1,23.00,1396.0,90.00,5.0
4,Maruti,2007,130000,120000,Petrol,Individual,Manual,1,16.10,1298.0,88.20,5.0


## Features, target and the train test split

In [4]:
from sklearn.model_selection import train_test_split

numeric_features = ['year', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'owner']
categorical_features = ['brand', 'fuel', 'seller_type', 'transmission']
target = 'selling_price'

# Check that no column was forgotten.
assert set(numeric_features + categorical_features + [target]) == set(df.columns)

X = df[numeric_features + categorical_features].copy()

# The raw selling price (not log). The class label is built from it in the next step.
price = df[target]

# Same 80/20 split and seed as A2.
X_train, X_test, price_train, price_test = train_test_split(
    X, price, test_size=0.2, random_state=RANDOM_STATE
)

## Price classes

The price is cut into 4 classes using the 25%, 50% and 75% quantiles.
The edges come from the training set only, so the test set does not influence them.

In [5]:
# Three edges that split the training prices into four groups of about equal size.
price_edges = price_train.quantile([0.25, 0.50, 0.75]).tolist()

# The outer bins go to -inf and +inf, so any test price (even a cheaper or dearer one
# than anything in training) still gets a class.
bins = [-np.inf] + price_edges + [np.inf]

# Class 0 is the cheapest group and class 3 is the most expensive.
y_train = pd.cut(price_train, bins=bins, labels=[0, 1, 2, 3]).astype(int)
y_test = pd.cut(price_test, bins=bins, labels=[0, 1, 2, 3]).astype(int)

print("Price edges (25%, 50%, 75%):", price_edges)
print()
print("Class counts in train:")
print(y_train.value_counts().sort_index())
print()
print("Class counts in test:")
print(y_test.value_counts().sort_index())

Price edges (25%, 50%, 75%): [250000.0, 415000.0, 650000.0]

Class counts in train:
selling_price
0    1480
1    1266
2    1450
3    1265
Name: count, dtype: int64

Class counts in test:
selling_price
0    366
1    345
2    375
3    280
Name: count, dtype: int64


## Preprocessing pipeline

This is the same pipeline as A2 (the version used for the from scratch model):
impute missing values, scale the numbers, one hot encode the categories.
It is fitted on the training data only.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# A2 filled roughly symmetric features with the mean and skewed ones with the median.
# The skewness is measured on the training set only.
train_skew = X_train[numeric_features].skew()
symmetric_features = train_skew[train_skew.abs() < 0.5].index.tolist()
skewed_features = train_skew[train_skew.abs() >= 0.5].index.tolist()

symmetric_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', StandardScaler())
])

skewed_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

# drop='first' avoids duplicate information between the one hot columns, which suits a
# model written from scratch. Unseen categories (like Ashok) do not crash the encoder.
categorical_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first',
                             handle_unknown='infrequent_if_exist',
                             sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('symmetric', symmetric_pipe, symmetric_features),
        ('skewed', skewed_pipe, skewed_features),
        ('categorical', categorical_pipe, categorical_features)
    ],
    remainder='drop'   # anything not listed is dropped
)

print("Mean fill and scale  :", symmetric_features)
print("Median fill and scale:", skewed_features)
print("Fill and one hot     :", categorical_features)

Mean fill and scale  : ['mileage']
Median fill and scale: ['year', 'km_driven', 'engine', 'max_power', 'seats', 'owner']
Fill and one hot     : ['brand', 'fuel', 'seller_type', 'transmission']


In [7]:
# Learn the fill values, scaling and categories from the training data only,
# then apply them unchanged to the test data. This prevents data leakage.
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)
feature_names = list(preprocessor.get_feature_names_out())

print("Shapes")
print("  X_train:", X_train_prep.shape)
print("  X_test :", X_test_prep.shape)
print("  y_train:", y_train.shape)
print("  y_test :", y_test.shape)

Shapes
  X_train: (5461, 41)
  X_test : (1366, 41)
  y_train: (5461,)
  y_test : (1366,)


c:\Users\tisab\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


## Task 1. Logistic Regression from scratch

The class is based on `02 - Multinomial Logistic Regression.ipynb`. Changes from that notebook:
- The bias (intercept) column is added inside the class, so the caller passes plain features.
- The labels are integers 0 to 3, and the class turns them into one hot rows itself.
- The softmax subtracts the row maximum first, so large numbers cannot overflow.
- Only batch gradient descent is kept, to keep the class short.

The class is saved to `app/logistic_regression.py`, because the web app will import the same file.

In [8]:
import os

# %%writefile does not create folders, so make sure app/ exists first.
os.makedirs("app", exist_ok=True)

In [9]:
%%writefile app/logistic_regression.py
import numpy as np


class RidgePenalty:
    """Ridge (L2) penalty: lambda * sum of squared weights, and its derivative."""

    def __init__(self, l):
        self.l = l  # lambda value, the strength of the penalty

    def __call__(self, W):
        # lambda * sum(w^2). Squaring makes every weight cost something, and big weights
        # cost much more than small ones, so the model prefers many small weights.
        return self.l * np.sum(np.square(W))

    def derivation(self, W):
        # The derivative of lambda * w^2 is 2 * lambda * w. It always points the same way as
        # the weight, so subtracting it pulls every weight towards zero.
        return self.l * 2 * W


class LogisticRegression:
    """Multinomial logistic regression (softmax) trained with batch gradient descent."""

    def __init__(self, k=4, lr=0.1, num_iter=1000, print_every=100,
                 use_penalty=False, lambda_=0.0):
        self.k = k                      # number of classes
        self.lr = lr                    # learning rate, the size of each update step
        self.num_iter = num_iter        # how many gradient descent steps to take
        self.print_every = print_every  # print the loss every this many steps (0 = never)

        # Ridge (L2) penalty. It is off by default, and then nothing below changes.
        # lambda_ has a trailing underscore because lambda is a Python keyword.
        #
        # Convention used here:
        #   loss = mean cross entropy over the m samples + lambda_ * sum(weights ** 2)
        #   grad = X.T @ (h - Y) / m               + 2 * lambda_ * weights
        # The cross entropy part is averaged over the samples, but the penalty is NOT divided
        # by m. So lambda_ is measured on the scale of the average loss: lambda_ = 0.1 means
        # "0.1 times the squared weights, next to an average loss per car". The assignment
        # formula adds lambda * sum(theta^2) to a loss that is summed (not averaged) over
        # the samples, so the same strength there would be lambda_ * m.
        # The bias weights (the first row of W) are never penalized: the bias only moves
        # the scores up or down and does not make the model more complex or more wiggly.
        if lambda_ < 0:
            raise ValueError("lambda_ must not be negative")
        self.use_penalty = use_penalty
        self.lambda_ = lambda_
        self.penalty = RidgePenalty(lambda_)

    def _add_bias(self, X):
        # The bias is the intercept. A column of ones lets it be learned as one more weight,
        # so the model is X @ W instead of X @ W + b. Doing it here means the caller never
        # has to remember it, and predict adds it in exactly the same way as fit.
        return np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)

    def _one_hot(self, y):
        # y holds class numbers such as [2, 0, 3]. The loss and the gradient need one row per
        # sample with a 1 in the true class column, e.g. 2 -> [0, 0, 1, 0].
        Y = np.zeros((y.shape[0], self.k))
        Y[np.arange(y.shape[0]), y] = 1
        return Y

    def softmax(self, scores):
        # Softmax turns the k raw scores of each sample into k probabilities that are all
        # positive (because of exp) and add up to 1 (because of the division).
        # Subtracting the row maximum first does not change the result, but it stops exp()
        # from overflowing when a score is large.
        shifted = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    def cross_entropy(self, Y, h):
        # Cross entropy loss: -mean( sum over classes of Y * log(h) ).
        # Y is 1 only for the true class, so each sample contributes -log(probability given
        # to the true class). A confident right answer costs almost 0, and a confident wrong
        # answer costs a lot. The clip keeps log() away from exactly 0.
        h = np.clip(h, 1e-15, 1.0)
        return -np.sum(Y * np.log(h)) / Y.shape[0]

    def fit(self, X, y):
        X = self._add_bias(np.asarray(X, dtype=float))
        y = np.asarray(y).astype(int)
        if y.min() < 0 or y.max() >= self.k:
            raise ValueError(f"y must contain integer labels from 0 to {self.k - 1}")

        m, n = X.shape                  # m samples, n weights per class (features + bias)
        Y = self._one_hot(y)            # shape (m, k)

        # One column of weights per class, shape (n, k). Starting from zeros is fine here:
        # this loss has a single bottom, so every start ends up at the same answer.
        self.W = np.zeros((n, self.k))
        self.losses = []                # the loss that is minimized (includes the penalty)
        self.ce_losses = []             # the cross entropy part only

        for i in range(self.num_iter):
            h = self.softmax(X @ self.W)             # predicted probabilities, shape (m, k)
            ce_loss = self.cross_entropy(Y, h)
            self.ce_losses.append(ce_loss)

            loss = ce_loss
            if self.use_penalty:
                # W[1:] skips the first row, which holds the bias weights.
                loss = ce_loss + self.penalty(self.W[1:])
            self.losses.append(loss)

            # Gradient of the loss with respect to W. Softmax and cross entropy cancel
            # nicely, leaving (h - Y): the gap between predicted and true probabilities.
            # X.T @ (h - Y) adds that gap up for every feature, and dividing by m takes
            # the mean, so the step size does not depend on the number of samples.
            grad = X.T @ (h - Y) / m

            if self.use_penalty:
                # The penalty adds its own derivative, 2 * lambda_ * w, to the gradient of every
                # weight except the bias. So each step also pulls the weights towards zero.
                grad[1:] = grad[1:] + self.penalty.derivation(self.W[1:])

            # Move the weights a small step against the gradient to lower the loss.
            self.W = self.W - self.lr * grad

            if self.print_every and i % self.print_every == 0:
                print(f"Loss at iteration {i}: {loss:.4f}")

        return self

    def predict_proba(self, X):
        # Probability of each of the k classes for every row, shape (m, k).
        X = self._add_bias(np.asarray(X, dtype=float))
        return self.softmax(X @ self.W)

    def predict(self, X):
        # The predicted class is the one with the highest probability.
        return np.argmax(self.predict_proba(X), axis=1)

    # ------------------------------------------------------------------------------------
    # Classification metrics, written with plain numpy.
    # y_true and y_pred are 1D arrays of class numbers 0 to k-1. None of these methods use
    # the trained weights, so they also work on any pair of label arrays.
    # ------------------------------------------------------------------------------------

    def _counts(self, y_true, y_pred, c):
        # For one class c, every car falls in exactly one of these groups. Class c is treated
        # as "positive" and every other class as "negative".
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        tp = np.sum((y_pred == c) & (y_true == c))   # true positive: said c, and it is c
        fp = np.sum((y_pred == c) & (y_true != c))   # false positive: said c, but it is not c
        fn = np.sum((y_pred != c) & (y_true == c))   # false negative: said not c, but it is c
        return tp, fp, fn

    def accuracy(self, y_true, y_pred):
        # correct predictions / all predictions. One number for the whole model, so it does
        # not show which class the mistakes are in. That is why the per class scores exist.
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        return float(np.sum(y_true == y_pred) / y_true.shape[0])

    def precision(self, y_true, y_pred, c):
        # TP / (TP + FP): of all the cars the model called class c, how many really are c.
        # TP + FP is the number of times the model predicted c. If it never predicted c the
        # division would be 0/0, so we return 0 instead.
        tp, fp, fn = self._counts(y_true, y_pred, c)
        if tp + fp == 0:
            return 0.0
        return float(tp / (tp + fp))

    def recall(self, y_true, y_pred, c):
        # TP / (TP + FN): of all the cars that really are class c, how many the model found.
        # TP + FN is the number of true class c cars. If there are none, return 0.
        tp, fp, fn = self._counts(y_true, y_pred, c)
        if tp + fn == 0:
            return 0.0
        return float(tp / (tp + fn))

    def f1(self, y_true, y_pred, c):
        # 2 * P * R / (P + R): the harmonic mean of precision and recall. It is high only when
        # both are high, so a model cannot hide a very low recall behind a high precision.
        # If both are 0 the division would be 0/0, so we return 0.
        p = self.precision(y_true, y_pred, c)
        r = self.recall(y_true, y_pred, c)
        if p + r == 0:
            return 0.0
        return float(2 * p * r / (p + r))

    # Macro average: the plain mean over the k classes, so every class counts the same no
    # matter how many cars it has.

    def macro_precision(self, y_true, y_pred):
        return float(np.mean([self.precision(y_true, y_pred, c) for c in range(self.k)]))

    def macro_recall(self, y_true, y_pred):
        return float(np.mean([self.recall(y_true, y_pred, c) for c in range(self.k)]))

    def macro_f1(self, y_true, y_pred):
        return float(np.mean([self.f1(y_true, y_pred, c) for c in range(self.k)]))

    # Weighted average: each class score is multiplied by the share of true cars in that
    # class (its support / all cars) before adding. Big classes count more, which matches
    # the overall performance better when the classes are not the same size.

    def _weights(self, y_true):
        y_true = np.asarray(y_true)
        return np.array([np.sum(y_true == c) for c in range(self.k)]) / y_true.shape[0]

    def weighted_precision(self, y_true, y_pred):
        scores = np.array([self.precision(y_true, y_pred, c) for c in range(self.k)])
        return float(np.sum(self._weights(y_true) * scores))

    def weighted_recall(self, y_true, y_pred):
        scores = np.array([self.recall(y_true, y_pred, c) for c in range(self.k)])
        return float(np.sum(self._weights(y_true) * scores))

    def weighted_f1(self, y_true, y_pred):
        scores = np.array([self.f1(y_true, y_pred, c) for c in range(self.k)])
        return float(np.sum(self._weights(y_true) * scores))


Overwriting app/logistic_regression.py


### Import the class from the file

In [10]:
from app.logistic_regression import LogisticRegression

print(LogisticRegression)

<class 'app.logistic_regression.LogisticRegression'>


### Train and check the accuracy

In [11]:
# The class needs dense numpy arrays and integer labels 0 to 3.
# I use new names so the X_train and y_train above (used by the preprocessing) stay as they are.
# X_train_prep and X_test_prep are the preprocessed versions of X_train and X_test.
X_train_np = np.asarray(X_train_prep, dtype=float)
X_test_np = np.asarray(X_test_prep, dtype=float)
y_train_np = y_train.to_numpy().astype(int)
y_test_np = y_test.to_numpy().astype(int)

assert isinstance(X_train_np, np.ndarray) and isinstance(X_test_np, np.ndarray)
assert not np.isnan(X_train_np).any() and not np.isnan(X_test_np).any()
assert set(np.unique(y_train_np)) == {0, 1, 2, 3}

model = LogisticRegression(k=4, lr=0.1, num_iter=1000, print_every=100)
model.fit(X_train_np, y_train_np)

# Plain accuracy: the share of cars whose predicted class is the true class.
train_accuracy = np.mean(model.predict(X_train_np) == y_train_np)
test_accuracy = np.mean(model.predict(X_test_np) == y_test_np)

print()
print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")

Loss at iteration 0: 1.3863
Loss at iteration 100: 0.8593
Loss at iteration 200: 0.7842
Loss at iteration 300: 0.7462
Loss at iteration 400: 0.7220
Loss at iteration 500: 0.7047
Loss at iteration 600: 0.6916
Loss at iteration 700: 0.6813
Loss at iteration 800: 0.6728
Loss at iteration 900: 0.6656

Train accuracy: 0.7378
Test accuracy : 0.7284


## Classification metrics

The metrics are now methods of the class (see `app/logistic_regression.py`). They are written with plain numpy from TP, FP and FN, then compared with scikit-learn's `classification_report`.

The helper below computes every score with my class, gets the same scores from scikit-learn, prints them side by side and checks them with `np.allclose`.

In [12]:
from sklearn.metrics import classification_report

CLASSES = [0, 1, 2, 3]

def compare_with_sklearn(model, y_true, y_pred):
    """Print my metrics next to scikit-learn's and check that they match."""
    # labels=CLASSES makes scikit-learn report all 4 classes even if one is missing.
    # zero_division=0 gives 0 (not a warning) when a score would be 0/0, like my class does.
    report = classification_report(y_true, y_pred, labels=CLASSES, output_dict=True, zero_division=0)

    # scikit-learn calls the f1 column "f1-score", the other two keep their names.
    sk_name = {"precision": "precision", "recall": "recall", "f1": "f1-score"}

    rows = [("accuracy", model.accuracy(y_true, y_pred), report["accuracy"])]
    for name in ["precision", "recall", "f1"]:
        my_function = getattr(model, name)
        for c in CLASSES:
            rows.append((f"{name} class {c}", my_function(y_true, y_pred, c), report[str(c)][sk_name[name]]))
    for average, sk_key in [("macro", "macro avg"), ("weighted", "weighted avg")]:
        for name in ["precision", "recall", "f1"]:
            my_function = getattr(model, f"{average}_{name}")
            rows.append((f"{average} {name}", my_function(y_true, y_pred), report[sk_key][sk_name[name]]))

    table = pd.DataFrame(rows, columns=["metric", "mine", "sklearn"]).set_index("metric")
    table["difference"] = (table["mine"] - table["sklearn"]).abs()
    print(table.round(4).to_string())
    print()
    print("All scores match sklearn (np.allclose):", np.allclose(table["mine"], table["sklearn"]))
    return table

### a. Mock data, small enough to check by hand

20 labels with unequal classes: 8 cars in class 0, 6 in class 1, 4 in class 2 and 2 in class 3.

| class | support | TP | FP | FN | precision = TP/(TP+FP) | recall = TP/(TP+FN) |
|---:|---:|---:|---:|---:|:---|:---|
| 0 | 8 | 6 | 1 | 2 | 6/7 = 0.857 | 6/8 = 0.75 |
| 1 | 6 | 4 | 1 | 2 | 4/5 = 0.8 | 4/6 = 0.667 |
| 2 | 4 | 3 | 3 | 1 | 3/6 = 0.5 | 3/4 = 0.75 |
| 3 | 2 | 1 | 1 | 1 | 1/2 = 0.5 | 1/2 = 0.5 |

Accuracy by hand is (6 + 4 + 3 + 1) / 20 = 14 / 20 = 0.7.

In [13]:
# 8 + 6 + 4 + 2 = 20 true labels.
mock_true = np.array([0]*8 + [1]*6 + [2]*4 + [3]*2)

# Predictions chosen to give the TP, FP and FN counts in the table above.
mock_pred = np.array([0]*6 + [1] + [2]      # true class 0: 6 right, 1 called class 1, 1 called class 2
                     + [1]*4 + [0] + [2]    # true class 1: 4 right, 1 called class 0, 1 called class 2
                     + [2]*3 + [3]          # true class 2: 3 right, 1 called class 3
                     + [3] + [2])           # true class 3: 1 right, 1 called class 2

# A model object is only needed to call the metric methods, so no training happens here.
metrics_model = LogisticRegression(k=4)

print(classification_report(mock_true, mock_pred, labels=CLASSES, zero_division=0))
mock_table = compare_with_sklearn(metrics_model, mock_true, mock_pred)

              precision    recall  f1-score   support

           0       0.86      0.75      0.80         8
           1       0.80      0.67      0.73         6
           2       0.50      0.75      0.60         4
           3       0.50      0.50      0.50         2

    accuracy                           0.70        20
   macro avg       0.66      0.67      0.66        20
weighted avg       0.73      0.70      0.71        20

                      mine  sklearn  difference
metric                                         
accuracy            0.7000   0.7000         0.0
precision class 0   0.8571   0.8571         0.0
precision class 1   0.8000   0.8000         0.0
precision class 2   0.5000   0.5000         0.0
precision class 3   0.5000   0.5000         0.0
recall class 0      0.7500   0.7500         0.0
recall class 1      0.6667   0.6667         0.0
recall class 2      0.7500   0.7500         0.0
recall class 3      0.5000   0.5000         0.0
f1 class 0          0.8000   0.8000  

### b. Real test predictions from the trained model

In [14]:
# y_test_np and model come from the training cell above.
test_pred = model.predict(X_test_np)

print(classification_report(y_test_np, test_pred, labels=CLASSES, zero_division=0))
test_table = compare_with_sklearn(model, y_test_np, test_pred)

              precision    recall  f1-score   support

           0       0.82      0.88      0.85       366
           1       0.67      0.55      0.61       345
           2       0.64      0.69      0.66       375
           3       0.78      0.80      0.79       280

    accuracy                           0.73      1366
   macro avg       0.73      0.73      0.73      1366
weighted avg       0.73      0.73      0.72      1366

                      mine  sklearn  difference
metric                                         
accuracy            0.7284   0.7284         0.0
precision class 0   0.8193   0.8193         0.0
precision class 1   0.6678   0.6678         0.0
precision class 2   0.6441   0.6441         0.0
precision class 3   0.7812   0.7812         0.0
recall class 0      0.8798   0.8798         0.0
recall class 1      0.5536   0.5536         0.0
recall class 2      0.6853   0.6853         0.0
recall class 3      0.8036   0.8036         0.0
f1 class 0          0.8485   0.8485  

### What does support mean?

In the classification report, **support** is the number of true samples of that class in `y_true`. For example, a support of 366 for class 0 means the test set has 366 cars whose real class is 0. It is counted from the true labels, so it does not change with the model's predictions.

Support matters for two reasons. It is the weight in the weighted average (a class's weight is its support divided by the total). It also shows how much to trust a score: a class with a small support, like class 3 in the mock data, can change a lot if just one car is predicted differently.

## Task 2. Ridge (L2) penalty

The class now has two more parameters, `use_penalty` (default `False`) and `lambda_` (default `0.0`). The convention is written in the class, in short:

- loss = mean cross entropy + `lambda_` * sum of squared weights
- gradient = the old gradient + 2 * `lambda_` * weights
- the bias weights are not penalized
- `lambda_` is not divided by the number of samples, so it is on the scale of the average loss

The style follows `RidgePenalty` in `Regularization.ipynb`.

### a. Penalty off: same results as before

In [15]:
# Same learning rate and iterations as the training cell, penalty left off (the default).
off_model = LogisticRegression(k=4, lr=0.1, num_iter=1000, print_every=0)
off_model.fit(X_train_np, y_train_np)

off_train_accuracy = np.mean(off_model.predict(X_train_np) == y_train_np)
off_test_accuracy = np.mean(off_model.predict(X_test_np) == y_test_np)
print(f"Train accuracy: {off_train_accuracy:.4f}")
print(f"Test accuracy : {off_test_accuracy:.4f}")

# The accuracies must be the ones from before, and the weights must be identical to the
# weights of the model trained in the training cell (same class, same settings).
assert round(off_train_accuracy, 4) == 0.7378 and round(off_test_accuracy, 4) == 0.7284
assert np.array_equal(off_model.W, model.W)
print("Penalty off gives exactly the same weights and accuracy as before.")

Train accuracy: 0.7378
Test accuracy : 0.7284
Penalty off gives exactly the same weights and accuracy as before.


### b. Penalty on: three values of lambda_

In [16]:
results = []

# The first row is the model without the penalty, so the shrinking is easy to see.
results.append({
    "lambda_": 0.0,
    "weight norm": np.linalg.norm(off_model.W[1:]),
    "final loss": off_model.losses[-1],
    "cross entropy": off_model.ce_losses[-1],
    "train accuracy": off_train_accuracy,
    "test accuracy": off_test_accuracy,
})

for lambda_ in [0.01, 0.1, 1.0]:
    ridge_model = LogisticRegression(k=4, lr=0.1, num_iter=1000, print_every=0,
                                     use_penalty=True, lambda_=lambda_)
    ridge_model.fit(X_train_np, y_train_np)
    results.append({
        "lambda_": lambda_,
        # W[1:] leaves out the bias row, so this is the size of the real feature weights only.
        "weight norm": np.linalg.norm(ridge_model.W[1:]),
        # final loss = cross entropy + penalty, which is what gradient descent minimizes.
        "final loss": ridge_model.losses[-1],
        "cross entropy": ridge_model.ce_losses[-1],
        "train accuracy": np.mean(ridge_model.predict(X_train_np) == y_train_np),
        "test accuracy": np.mean(ridge_model.predict(X_test_np) == y_test_np),
    })

ridge_table = pd.DataFrame(results).set_index("lambda_")
print(ridge_table.round(4).to_string())

# The weight norm should shrink as lambda_ grows.
norms = ridge_table["weight norm"].to_numpy()
print()
print("Weight norm shrinks as lambda_ grows:", bool(np.all(np.diff(norms) < 0)))

         weight norm  final loss  cross entropy  train accuracy  test accuracy
lambda_                                                                       
0.00          5.6075      0.6596         0.6596          0.7378         0.7284
0.01          3.1368      0.8562         0.7578          0.7189         0.7167
0.10          1.0635      1.1132         1.0000          0.6215         0.6149
1.00          0.2313      1.3176         1.2641          0.5541         0.5549

Weight norm shrinks as lambda_ grows: True


### Why the penalty shrinks the weights

Without the penalty, the model only tries to make the training predictions right, and a big weight is free. With the penalty, every big weight adds to the loss, so the model has to pay for it. A big weight is only kept when it lowers the prediction error by more than it costs.

You can also see it in the update step. The penalty adds 2 * `lambda_` * weight to the gradient, so each step first pulls the weight a little towards zero, and then the prediction error pushes it back if it really needs to be big. A larger `lambda_` means a stronger pull and smaller weights.

Small weights make the model change less when a feature changes, which is why the penalty helps against overfitting. If `lambda_` is too large, the weights become too small for the model to learn the pattern, and the accuracy goes down.

## Baseline run in a local MLflow

The course MLflow server is returning 404, so this run goes to a local SQLite store. To use the course server later, change `MLFLOW_URI` at the top of the next cell (the course address is in the comment above it). That is the only place the address is written.

The run logs the parameters, the metrics (test set, except the final loss and the train accuracy), the model with a signature, and the fitted preprocessing pipeline and `price_edges` as small artifacts. The dataset itself is not logged.

In [17]:
# Tracking address. To use the course server, swap to: "http://mlflow.ml.brain.cs.ait.ac.th/"
MLFLOW_URI = "sqlite:///mlflow.db"

import os
import tempfile

import joblib
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# The course server shows this as the run's user.
os.environ["LOGNAME"] = "st126686"

mlflow.set_tracking_uri(MLFLOW_URI)
experiment = mlflow.set_experiment("st126686-a3")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment  :", experiment.name)

baseline_params = {"lr": 0.1, "num_iter": 1000, "use_penalty": False, "lambda_": 0.0}
baseline = LogisticRegression(k=4, print_every=0, **baseline_params)

with mlflow.start_run(run_name="baseline") as run:
    mlflow.log_params(baseline_params)

    baseline.fit(X_train_np, y_train_np)
    baseline_train_pred = baseline.predict(X_train_np)
    baseline_test_pred = baseline.predict(X_test_np)

    # Same settings as the training cell, so the weights must be the same.
    assert np.array_equal(baseline.W, model.W)

    # final_loss and train_accuracy come from the training set. Everything else is the test set.
    baseline_metrics = {
        "final_loss": baseline.losses[-1],
        "train_accuracy": baseline.accuracy(y_train_np, baseline_train_pred),
        "test_accuracy": baseline.accuracy(y_test_np, baseline_test_pred),
    }
    for average in ["macro", "weighted"]:
        for name in ["precision", "recall", "f1"]:
            my_function = getattr(baseline, f"{average}_{name}")
            baseline_metrics[f"{average}_{name}"] = my_function(y_test_np, baseline_test_pred)
    for c in CLASSES:
        baseline_metrics[f"f1_class_{c}"] = baseline.f1(y_test_np, baseline_test_pred, c)
    mlflow.log_metrics({name: float(value) for name, value in baseline_metrics.items()})

    # The signature only describes the input and output types and shapes, not the data itself.
    signature = infer_signature(X_test_np, baseline_test_pred)
    # The default format (skops) refuses my own class as an untrusted type, so save with cloudpickle.
    mlflow.sklearn.log_model(
        baseline,
        name="model",
        signature=signature,
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )

    # Small artifacts so the web app can rebuild the full prediction:
    # raw columns -> preprocessor -> model -> class number, and price_edges gives the class ranges.
    with tempfile.TemporaryDirectory() as tmp_dir:
        preprocessor_path = os.path.join(tmp_dir, "preprocessor.joblib")
        joblib.dump(preprocessor, preprocessor_path)
        mlflow.log_artifact(preprocessor_path, artifact_path="preprocessing")
    mlflow.log_dict({"price_edges": price_edges}, "preprocessing/price_edges.json")

run_id = run.info.run_id
print()
print(pd.Series(baseline_metrics).round(4).to_string())
print()
print("Run id:", run_id)

Tracking URI: sqlite:///mlflow.db
Experiment  : st126686-a3


2026/10/01 17:53:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



final_loss            0.6596
train_accuracy        0.7378
test_accuracy         0.7284
macro_precision       0.7281
macro_recall          0.7306
macro_f1              0.7276
weighted_precision    0.7252
weighted_recall       0.7284
weighted_f1           0.7249
f1_class_0            0.8485
f1_class_1            0.6054
f1_class_2            0.6641
f1_class_3            0.7923

Run id: 8cb98c6242a8457a8c77b5c6fe76f81d


## Experiment grid

16 runs: learning rate (2 values) x iterations (2 values) x penalty (off or 3 values of `lambda_`). Every run is logged to the same local MLflow and the same experiment as the baseline, through `MLFLOW_URI`.

The configuration is chosen on a validation part cut from the training set, never on the test set. The test scores are logged too, only to look at afterwards.

### Training part and validation part

The 80/20 split is stratified by class, so the four price classes keep the same shares in both parts.

In [18]:
from sklearn.model_selection import train_test_split

# 80% to train on, 20% to compare the settings on. The test set is not used here.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_np, y_train_np, test_size=0.2, stratify=y_train_np, random_state=RANDOM_STATE
)

print("Training part  :", X_fit.shape, "| validation part:", X_val.shape)
print()
print("Share of each class")
print(pd.DataFrame({
    "training part": np.bincount(y_fit) / len(y_fit),
    "validation part": np.bincount(y_val) / len(y_val),
}).round(3).to_string())

Training part  : (4368, 41) | validation part: (1093, 41)

Share of each class
   training part  validation part
0          0.271            0.271
1          0.232            0.232
2          0.266            0.265
3          0.232            0.231


In [19]:
import logging
import tempfile
from itertools import product

import joblib
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Same local MLflow and experiment as the baseline run (MLFLOW_URI is set in the cell above).
os.environ["LOGNAME"] = "st126686"
mlflow.set_tracking_uri(MLFLOW_URI)
experiment = mlflow.set_experiment("st126686-a3")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment  :", experiment.name)

learning_rates = [0.1, 0.5]
iteration_counts = [1000, 3000]
penalty_lambdas = [None, 0.0001, 0.001, 0.01]   # None means the penalty is off

# MLflow repeats the cloudpickle warning for every model. It was shown once for the baseline.
logging.getLogger("mlflow.sklearn").setLevel(logging.ERROR)

grid_records = []
problem_lines = []

with tempfile.TemporaryDirectory() as tmp_dir:
    # The fitted preprocessing pipeline is the same for every run, so it is saved once here.
    preprocessor_path = os.path.join(tmp_dir, "preprocessor.joblib")
    joblib.dump(preprocessor, preprocessor_path)

    for lr, num_iter, penalty_lambda in product(learning_rates, iteration_counts, penalty_lambdas):
        use_penalty = penalty_lambda is not None
        lambda_ = penalty_lambda if use_penalty else 0.0
        run_name = f"lr{lr}_it{num_iter}_" + (f"l{lambda_}" if use_penalty else "off")
        params = {"lr": lr, "num_iter": num_iter, "use_penalty": use_penalty, "lambda_": lambda_}

        grid_model = LogisticRegression(k=4, print_every=0, **params)

        with mlflow.start_run(run_name=run_name) as grid_run:
            mlflow.log_params(params)

            # Train on the training part only.
            grid_model.fit(X_fit, y_fit)
            fit_pred = grid_model.predict(X_fit)
            val_pred = grid_model.predict(X_val)
            test_pred_grid = grid_model.predict(X_test_np)

            # Check the loss curve. A tiny rise from rounding is ignored (1e-10).
            loss_curve = np.array(grid_model.losses)
            has_nan = not np.all(np.isfinite(loss_curve))
            n_rises = int(np.sum(np.diff(loss_curve) > 1e-10))
            loss_check = "nan" if has_nan else ("loss went up" if n_rises > 0 else "ok")
            mlflow.set_tag("loss_check", loss_check)
            if loss_check != "ok":
                problem_lines.append(f"{run_name}: {loss_check} ({n_rises} rises, final loss {loss_curve[-1]})")

            # final_loss and train_accuracy are on the training part, val_* on the validation
            # part, test_* on the test set (logged to look at later, not used to choose).
            metrics = {
                "final_loss": loss_curve[-1],
                "train_accuracy": grid_model.accuracy(y_fit, fit_pred),
                "val_accuracy": grid_model.accuracy(y_val, val_pred),
                "val_macro_f1": grid_model.macro_f1(y_val, val_pred),
                "val_weighted_f1": grid_model.weighted_f1(y_val, val_pred),
                "test_accuracy": grid_model.accuracy(y_test_np, test_pred_grid),
                "test_macro_f1": grid_model.macro_f1(y_test_np, test_pred_grid),
                "test_weighted_f1": grid_model.weighted_f1(y_test_np, test_pred_grid),
            }
            for c in CLASSES:
                metrics[f"f1_class_{c}"] = grid_model.f1(y_val, val_pred, c)
            metrics = {name: float(value) for name, value in metrics.items()}
            mlflow.log_metrics(metrics)

            # Same logging as the baseline, so any run can be registered later.
            signature = infer_signature(X_test_np, test_pred_grid)
            mlflow.sklearn.log_model(
                grid_model,
                name="model",
                signature=signature,
                serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            )
            mlflow.log_artifact(preprocessor_path, artifact_path="preprocessing")
            mlflow.log_dict({"price_edges": price_edges}, "preprocessing/price_edges.json")

        grid_records.append({"run_name": run_name, "run_id": grid_run.info.run_id,
                             "loss_check": loss_check, **params, **metrics})

logging.getLogger("mlflow.sklearn").setLevel(logging.WARNING)

print()
print(f"Logged {len(grid_records)} runs.")
if problem_lines:
    print("Runs with a loss problem:")
    for line in problem_lines:
        print("  ", line)
else:
    print("No run had a NaN loss or a loss that went up.")

# All runs, best validation macro f1 first.
grid_table = (pd.DataFrame(grid_records)
              .sort_values("val_macro_f1", ascending=False, kind="stable")
              .reset_index(drop=True))
columns = ["run_name", "lr", "num_iter", "use_penalty", "lambda_",
           "val_macro_f1", "test_macro_f1", "test_accuracy"]
print()
print(grid_table[columns].round(4).to_string())

best_run = grid_table.iloc[0]
print()
print("Top run by val_macro_f1:", best_run["run_name"])
print("Run id:", best_run["run_id"])

Tracking URI: sqlite:///mlflow.db
Experiment  : st126686-a3

Logged 16 runs.
No run had a NaN loss or a loss that went up.

                run_name   lr  num_iter  use_penalty  lambda_  val_macro_f1  test_macro_f1  test_accuracy
0   lr0.5_it3000_l0.0001  0.5      3000         True   0.0001        0.7379         0.7348         0.7321
1   lr0.5_it1000_l0.0001  0.5      1000         True   0.0001        0.7378         0.7388         0.7365
2    lr0.5_it3000_l0.001  0.5      3000         True   0.0010        0.7372         0.7327         0.7306
3       lr0.5_it3000_off  0.5      3000        False   0.0000        0.7367         0.7361         0.7335
4       lr0.5_it1000_off  0.5      1000        False   0.0000        0.7361         0.7367         0.7343
5    lr0.5_it1000_l0.001  0.5      1000         True   0.0010        0.7353         0.7332         0.7313
6       lr0.1_it3000_off  0.1      3000        False   0.0000        0.7348         0.7346         0.7328
7   lr0.1_it3000_l0.0001  0.

A higher learning rate helped when training was short: with no penalty and 1000 iterations, `lr` 0.5 reached a validation macro f1 of 0.7361, against 0.7238 for `lr` 0.1. More iterations mostly rescued the slow learning rate (`lr` 0.1 rose to 0.7348 at 3000 iterations) and did almost nothing for `lr` 0.5 (0.7361 to 0.7367), even though its training loss kept falling from 0.5964 to 0.5828. A small penalty made no real difference (`lambda_` 0.0001 and 0.001 stay within about 0.003 of the run without a penalty at the same `lr` and iterations), but `lambda_` 0.01 was clearly worse, with a validation macro f1 of 0.7127 at best and a final loss stuck near 0.855, because it shrinks the weights too much for the model to fit. The top run, `lr0.5_it3000_l0.0001`, scored 0.7379 on validation and 0.7348 on the test set, but the six best runs are within 0.003 of each other on validation, so with 1093 validation cars that first place is not a real win over the others.

In [ ]:
# Register the best grid run in the local MLflow model registry (same MLFLOW_URI as above).
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient()

best_run_id = "4f2f4d454ad748abb14510d4bcf26106"   # lr0.5_it3000_l0.0001
registered_name = "st126686-a3-model"
print("Run name     :", client.get_run(best_run_id).info.run_name)

model_version = mlflow.register_model(f"runs:/{best_run_id}/model", registered_name)

# Stages are deprecated in MLflow 3, so the version is marked with the alias "staging" instead.
print("MLflow       :", mlflow.__version__)
client.set_registered_model_alias(registered_name, "staging", model_version.version)

staged = client.get_model_version_by_alias(registered_name, "staging")
print("Model name   :", staged.name)
print("Version      :", staged.version)
print("Aliases      :", staged.aliases)
print("Stage        :", staged.current_stage)

Run name     : lr0.5_it3000_l0.0001
MLflow       : 3.16.0
Model name   : st126686-a3-model
Version      : 1
Aliases      : ['staging']
Stage        : None


Successfully registered model 'st126686-a3-model'.
2026/10/01 18:07:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 4f2f4d454ad748abb14510d4bcf26106 has no artifacts at artifact path 'model', registering model based on models:/m-0de28ce4804e4cd0a21251d904a141c9 instead
Created version '1' of model 'st126686-a3-model'.


In [ ]:
# Export the registered model (alias "staging") and its preprocessing for the web app.
# app/pages/predict.py reads exactly these three files from model/ at the project root.
import json
import os
import pickle
import shutil

import joblib
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient()

export_dir = "model"
os.makedirs(export_dir, exist_ok=True)

staged = client.get_model_version_by_alias("st126686-a3-model", "staging")
print(f"Exporting {staged.name} version {staged.version} (run {staged.run_id})")

# model.pkl: the LogisticRegression instance itself, saved with plain pickle.
chosen_model = mlflow.sklearn.load_model(f"models:/{staged.name}@staging")
with open(os.path.join(export_dir, "model.pkl"), "wb") as f:
    pickle.dump(chosen_model, f)

# preprocessor.joblib and price_edges.json: copied unchanged from the same run.
run_files = mlflow.artifacts.download_artifacts(run_id=staged.run_id, artifact_path="preprocessing")
for name in ["preprocessor.joblib", "price_edges.json"]:
    shutil.copy(os.path.join(run_files, name), os.path.join(export_dir, name))

# Check: reload the three files the way the app does and score the test set again.
with open(os.path.join(export_dir, "model.pkl"), "rb") as f:
    exported_model = pickle.load(f)
exported_preprocessor = joblib.load(os.path.join(export_dir, "preprocessor.joblib"))
with open(os.path.join(export_dir, "price_edges.json")) as f:
    exported_edges = json.load(f)["price_edges"]

assert exported_edges == price_edges
exported_pred = exported_model.predict(exported_preprocessor.transform(X_test))
exported_accuracy = exported_model.accuracy(y_test.to_numpy(), exported_pred)
logged_accuracy = client.get_run(staged.run_id).data.metrics["test_accuracy"]
assert np.isclose(exported_accuracy, logged_accuracy)

print(f"Test accuracy of the exported files: {exported_accuracy:.4f} (logged: {logged_accuracy:.4f})")
print("Price edges:", exported_edges)
for name in sorted(os.listdir(export_dir)):
    print(f"  {export_dir}/{name}: {os.path.getsize(os.path.join(export_dir, name)):,} bytes")

Exporting st126686-a3-model version 1 (run 4f2f4d454ad748abb14510d4bcf26106)
Test accuracy of the exported files: 0.7321 (logged: 0.7321)
Price edges: [250000.0, 415000.0, 650000.0]
  model/model.pkl: 115,782 bytes
  model/preprocessor.joblib: 5,848 bytes
  model/price_edges.json: 73 bytes



C:\Users\tisab\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
